Copyright (c) Microsoft Corporation.
Licensed under the MIT License.

<p align="center">
  <img src="./assets/banners/02-give-it-a-voice.svg" alt="MAI Models · Multimodal Product Launch — Give it a voice: How can MAI Voice 2.1 models give a launch its voice? Microsoft AI Samples" width="900" />
</p>

# 🎙️ Give it a voice

> **How can MAI Voice 2.1 models give a product launch its voice — and how would you explain the choice to a customer?**

⏱️ About 10 minutes to review, or about 15 to run live · 🧱 You'll build: the launch audio for the TrailMaster X4

**By the end, you'll be able to:**

- Turn a script into speech with MAI Voice 2.1, in the right voice, language and tone
- Measure what matters for voice: how long the audio runs, and how long until the first sound
- Explain when to use MAI Voice 2.1 Flash instead

> 🧪 **Heads-up: you can read this one or run it.** In replay mode (the default) it plays saved audio from an earlier run, in `examples/02/`. Switch to live mode to make your own; every run sounds a little different.

> 💡 **How to use this notebook:** run each cell in order with **Shift + Enter**. Before each code cell, read the note and **take a guess** at what you'll hear.

<img src="./assets/divider.svg" alt="" width="100%" height="6" />

## 2.1 The story: the posters need a voice

In notebook 01, we made the TrailMaster X4's launch kit: posters, a catalog page, ads in three languages. Now marketing wants to *hear* it.

They need a radio spot, store announcements, the same ad for Mexico and France, a setup hotline that answers fast, the setup guide read aloud, and example calls to train new support staff.

That's six jobs for one voice model. Let's see how far it gets us, and where the faster version is the better pick.

### Pick your route

| Route | Sections | Good for |
| --- | --- | --- |
| ⚡ **Short** (about 5 minutes to review) | 2.4, 2.5, 2.8, 2.11 | A first listen before a customer meeting |
| 🧭 **Full** (about 10 minutes to review) | All of them | Learning every feature well enough to teach it |

<img src="./assets/divider.svg" alt="" width="100%" height="6" />

## 2.2 New words

| Word | In plain words |
| --- | --- |
| **Text to speech** | A model that reads text out loud |
| **Voice** | Who's speaking, like `en-US-Harper` or `es-MX-Valeria`. The same voice works with both models |
| **SSML** | A small markup language that tells the voice model what to say and how |
| **Speaking style** | The tone of a line, like `excited`, `narrator` or `customer_call_center` |
| **Time to first sound** | How long a listener waits before the voice starts. On a phone line, this is what people notice |
| **Flash version** | A faster version of the same model, built for low latency |

<img src="./assets/divider.svg" alt="" width="100%" height="6" />

## 2.3 Before you start

**Just reading?** Run the next cell, then go to 2.4: replay mode needs no Azure.

**Running it live?** Running `bash infra/setup.sh` from this folder sets you up:

| You need | Why |
| --- | --- |
| A Microsoft Foundry resource in a region with MAI voices | MAI Voice 2.1 and Flash come with it; there's nothing to deploy. `eastus` works ([regions](https://learn.microsoft.com/azure/ai-services/speech-service/regions)) |
| The **Foundry User** role, and a sign-in | Lets you use speech without a key. Run `az login` in a terminal |
| Live mode | Set `DEMO_MODE = "live"` in the next cell, or add `DEMO_MODE=live` to this folder's `.env` |

**What it costs.** You pay per character spoken. A full run reads a few thousand characters; see [Speech pricing](https://aka.ms/SpeechPricing).

> 🗣️ **Say when it's AI.** If people will hear these voices, tell them the voice is AI-generated. See the [text-to-speech transparency note](https://learn.microsoft.com/azure/ai-foundry/responsible-ai/speech-service/text-to-speech/transparency-note).

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data" / "contoso-retail").is_dir())
load_dotenv(ROOT / "workshops" / "mai-multimodal-product-launch" / ".env")

DEMO_MODE = None  # ✏️ Set to "live" or "replay" to override .env. Leave None to use .env.

mode = (DEMO_MODE or os.environ.get("DEMO_MODE") or "replay").strip().lower()
if mode not in ("replay", "live"):
    raise SystemExit(f'⚠️ DEMO_MODE is "{mode}". Use "replay" or "live".')
source = "this cell" if DEMO_MODE else ".env" if os.environ.get("DEMO_MODE") else "the default"
print(f"🎬 Mode: {'🔁 replay' if mode == 'replay' else '🎙️ live'} (from {source})")

🎬 Mode: 🎙️ live (from this cell)


<img src="./assets/divider.svg" alt="" width="100%" height="6" />

## 2.4 Get set up

Two small cells. The first loads your settings and signs you in **without a key**. The second holds our helper, `speak()`, which sends text to the voice model, saves the MP3, and measures how long the first sound took.

> 🔮 **Guess first:** which two model names will it print?

In [2]:
# Settings come from this folder's .env, which infra/setup.sh writes, so no names or secrets are written into the code.
import json
import os
import re
import statistics
import time
from html import escape
from pathlib import Path
from urllib.parse import urlparse

import azure.cognitiveservices.speech as speechsdk
from azure.identity import DefaultAzureCredential
from IPython.display import Audio, display
from dotenv import load_dotenv

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data" / "contoso-retail").is_dir())
HERE = ROOT / "workshops" / "mai-multimodal-product-launch"
DATA = ROOT / "data" / "contoso-retail"  # The shared product catalog: manuals, photos and products.json
load_dotenv(HERE / ".env")


def setting(name: str, default: str | None = None) -> str:
    """Return a setting from .env, or stop with a message that says how to fix it."""
    value = os.environ.get(name, "").strip()
    if value and "<" not in value:
        return value
    if default is not None:
        return default
    raise SystemExit(f"⚠️ {name} isn't set yet. Run `bash infra/setup.sh` (it writes .env), then run this cell again.")


def clean(text: str | None) -> str:
    """Keep error messages short, and free of IDs and addresses, before they land in saved output."""
    text = re.sub(r"[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}", "<id>", text or "")
    return re.sub(r"(?:https?|wss?)://\S+", "<address>", text)[:240]


# The mode cell in 2.3 wins over .env. Replay shows saved results and needs no Azure at all.
REPLAY = (globals().get("DEMO_MODE") or setting("DEMO_MODE", "replay")).strip().lower() == "replay"
OUTPUT = HERE / "output" / "02"      # Your results. Git ignores this folder.
EXAMPLES = HERE / "examples" / "02"  # Results from one of our runs, so anyone can replay the notebook
OUTPUT.mkdir(parents=True, exist_ok=True)

VOICE_HERO = setting("MODEL_VOICE_HERO", "MAI-Voice-2.1")        # Most expressive, for audio people will hear again and again
VOICE_FAST = setting("MODEL_VOICE_FAST", "MAI-Voice-2.1-Flash")  # Lower latency, for when someone is waiting
LOG_FILE = OUTPUT / "run_log.json"
run_log: dict[str, dict] = json.loads(LOG_FILE.read_text()) if LOG_FILE.exists() else {}
example_log: dict[str, dict] = json.loads((EXAMPLES / "run_log.json").read_text()) if (EXAMPLES / "run_log.json").exists() else {}

credential = DefaultAzureCredential()  # Keyless: uses your az login, not a secret key.

print(f"✅ Settings loaded. Best quality: {VOICE_HERO} · Faster: {VOICE_FAST}")
print(f"   Mode: {'🔁 replay (your audio first, then examples/02/)' if REPLAY else '🎙️ live'}")

✅ Settings loaded. Best quality: MAI-Voice-2.1 · Faster: MAI-Voice-2.1-Flash
   Mode: 🎙️ live


`speak(name, text, voice=…, model=…, style=…)` turns text into an MP3 in `output/02/`. It also saves every script, so notebook 03 can check what a listener would actually hear.

In [3]:
HINT = ("Check three things: you're signed in (az login), you have the Foundry User role on the "
        "Foundry resource, and your resource is in a region with MAI voices.")
SCRIPTS: dict[str, dict] = json.loads((OUTPUT / "scripts.json").read_text()) if (OUTPUT / "scripts.json").exists() else {}


def _speech_config() -> speechsdk.SpeechConfig:
    if not hasattr(_speech_config, "value"):
        # Speech lives on your Foundry resource's cognitiveservices address, worked out from the project endpoint.
        resource = urlparse(setting("FOUNDRY_PROJECT_ENDPOINT")).hostname.split(".")[0]
        config = speechsdk.SpeechConfig(token_credential=credential, endpoint=f"https://{resource}.cognitiveservices.azure.com/")
        config.set_speech_synthesis_output_format(speechsdk.SpeechSynthesisOutputFormat.Audio24Khz48KBitRateMonoMp3)
        _speech_config.value = config
    return _speech_config.value


def ssml(text: str, voice: str, model: str, style: str | None = None, lang: str = "en-US") -> str:
    """Wrap words in SSML: which voice, which model, and optionally which speaking style."""
    body = escape(text)
    if style:
        body = f'<mstts:express-as style="{style}">{body}</mstts:express-as>'
    return f'<voice name="{voice}:{model}">{body}</voice>'


def document(parts: list[str], lang: str = "en-US") -> str:
    return ('<speak version="1.0" xmlns="http://www.w3.org/2001/10/synthesis" '
            f'xmlns:mstts="http://www.w3.org/2001/mstts" xml:lang="{lang}">' + "".join(parts) + "</speak>")


def saved(name: str) -> Path | None:
    """Where saved audio lives: output/02/ (yours) first, then examples/02/ (ours)."""
    for path in (OUTPUT / f"{name}.mp3", EXAMPLES / f"{name}.mp3"):
        if path.exists():
            return path
    return None


def speak(name: str, text: str, voice: str = "en-US-Harper", model: str | None = None, style: str | None = None,
          lang: str = "en-US", parts: list[str] | None = None, turns: list[tuple[str, str]] | None = None) -> Path | None:
    """Turn text into speech, save it as MP3, and time it. Returns the file, or None in replay if nothing was saved."""
    model = model or VOICE_HERO
    SCRIPTS[name] = {"text": text, "lang": lang, "voice": voice, "model": model, "turns": turns or []}
    (OUTPUT / "scripts.json").write_text(json.dumps(SCRIPTS, indent=2, ensure_ascii=False))  # Notebook 03 checks what it hears against these.
    if REPLAY:
        path = saved(name)
        entry = run_log.get(name) if path and path.parent == OUTPUT else example_log.get(name)
        if not (path and entry):
            print(f"⏭️ {name}: no saved audio yet. Set DEMO_MODE=live to make it.")
            return None
        run_log[name] = entry
        print(f"🔁 {name} · {entry['model']} · first sound after {entry['first_audio']} s · {entry['audio_seconds']} s of audio")
        return path
    path = OUTPUT / f"{name}.mp3"
    synthesizer = speechsdk.SpeechSynthesizer(speech_config=_speech_config(), audio_config=speechsdk.audio.AudioOutputConfig(filename=str(path)))
    first: dict[str, float] = {}
    started = time.perf_counter()
    synthesizer.synthesizing.connect(lambda event: first.setdefault("seconds", time.perf_counter() - started))
    result = synthesizer.speak_ssml_async(document(parts or [ssml(text, voice, model, style, lang)], lang)).get()
    total = time.perf_counter() - started
    del synthesizer  # Closes the file, so the audio on disk is complete.
    if result.reason != speechsdk.ResultReason.SynthesizingAudioCompleted:
        details = result.cancellation_details
        raise SystemExit(f"⚠️ {model} didn't finish ({details.reason}). {HINT}\n{clean(details.error_details)}")
    entry = {"model": model, "voice": voice, "first_audio": round(first.get("seconds", total), 2),
             "total": round(total, 2), "audio_seconds": round(result.audio_duration.total_seconds(), 1)}
    run_log[name] = entry
    LOG_FILE.write_text(json.dumps({key: value for key, value in run_log.items() if (OUTPUT / f"{key}.mp3").exists()}, indent=2))
    print(f"🎙️ {name} · {model} · first sound after {entry['first_audio']} s · {entry['audio_seconds']} s of audio")
    return path


def play(clips: dict[str, Path | None]) -> None:
    """Show a player for each clip, with a label."""
    for label, path in clips.items():
        if path:
            print(f"▶️ {label}")
            display(Audio(filename=str(path)))

<img src="./assets/divider.svg" alt="" width="100%" height="6" />

## 2.5 Cold open: a 15-second radio spot

<img src="./assets/asks/02-05.svg" alt="Your customer asks: We need a 15-second radio spot. Can it sound like a real ad?" width="900" />
<br/>

Radio ads have a hard limit: the slot is 15 seconds, and not a second more. Here's our script, built from the poster in 01. Notice it says the tent comes *with a rainfly*, not that it's waterproof (see 0.1 for why).

> *Rain in the forecast? Bring it. The TrailMaster X4 from OutdoorLiving sleeps four, stands up on its own, and comes with its own rainfly. Home, anywhere, for $250. Find it at Contoso Outdoors.*

> 🔮 **Guess first:** will it fit in 15 seconds? And how will it say *X4* and *$250*?

In [4]:
RADIO_EN = "Rain in the forecast? Bring it. The TrailMaster X4 from OutdoorLiving sleeps four, stands up on its own, and comes with its own rainfly. Home, anywhere, for $250. Find it at Contoso Outdoors."

radio = speak("05-radio-en", RADIO_EN, voice="en-US-Harper", model=VOICE_HERO, style="excited")
play({"Radio spot · MAI-Voice-2.1 · Harper · excited": radio})
if radio:
    length = run_log["05-radio-en"]["audio_seconds"]
    print(f"\n⏱️ {length} seconds: {'✅ fits the 15-second slot' if length <= 15 else '❌ too long for the slot'}")

🎙️ 05-radio-en · MAI-Voice-2.1 · first sound after 1.68 s · 16.2 s of audio
▶️ Radio spot · MAI-Voice-2.1 · Harper · excited



⏱️ 16.2 seconds: ❌ too long for the slot


👀 **What to look for:**

- [ ] Does it fit in **15 seconds**?
- [ ] Does it say *"TrailMaster X four"* and *"two hundred fifty dollars"*, the way a person would?
- [ ] Does it sound like an **ad**, with energy, or like someone reading a list?

⚠️ **The honest limit:** if it runs long, cut words; don't ask the voice to rush. A rushed ad sounds cheap. Rewrite, then measure again.

🧪 **Try it yourself:** write a 10-second version for a shorter slot. Which words did you cut first?

<img src="./assets/divider.svg" alt="" width="100%" height="6" />

## 2.6 One voice, three moods

<img src="./assets/asks/02-06.svg" alt="Your customer asks: Our brand has one voice. Can it change its tone for each moment?" width="900" />
<br/>

A brand voice should be recognizable everywhere, but a store announcement shouldn't sound like a support call. Same voice, same words, three speaking styles:

> 🔮 **Guess first:** which style would you pick for an announcement over the store speakers?

In [5]:
LINE = "Welcome to Contoso Outdoors. The TrailMaster X4 is back, and it's ready for your next trip."

moods = {style: speak(f"06-mood-{style}", LINE, voice="en-US-Harper", model=VOICE_HERO, style=style)
         for style in ("narrator", "excited", "customer_call_center")}
play({f"Harper · {style}": path for style, path in moods.items()})

🎙️ 06-mood-narrator · MAI-Voice-2.1 · first sound after 2.51 s · 7.2 s of audio
🎙️ 06-mood-excited · MAI-Voice-2.1 · first sound after 2.15 s · 7.5 s of audio
🎙️ 06-mood-customer_call_center · MAI-Voice-2.1 · first sound after 1.56 s · 5.7 s of audio
▶️ Harper · narrator


▶️ Harper · excited


▶️ Harper · customer_call_center


👀 **What to look for:**

- [ ] Can you hear that it's the **same person** each time?
- [ ] Do the moods really differ: calm, upbeat, helpful?
- [ ] Which one would you play in the store? Which one on the hotline?

⚠️ **The honest limit:** each voice supports its own set of styles. Harper has many; some voices have only a few. Check the list in the [MAI voices docs](https://learn.microsoft.com/azure/ai-services/speech-service/mai-voices) before you design around one.

🧪 **Try it yourself:** try `hopeful` or `joyful`. Do they suit a tent?

<img src="./assets/divider.svg" alt="" width="100%" height="6" />

## 2.7 Around the world in three voices

<img src="./assets/asks/02-07.svg" alt="Your customer asks: We sell in Mexico and France too. Can the ad speak their language?" width="900" />
<br/>

In notebook 01 we drew the ad in English, Spanish and Japanese. Now let's make the radio spot for Mexico and France, with a native voice for each.

We translated these ourselves. **Have a native speaker check them** before anything airs.

> 🔮 **Guess first:** will the Spanish and French spots also fit in 15 seconds?

In [6]:
RADIO = {
    "es": ("es-MX", "es-MX-Valeria", "¿Lluvia en el pronóstico? Que venga. La TrailMaster X4 de OutdoorLiving es para cuatro personas, "
                                      "se sostiene sola y trae su propio doble techo. Tu casa, en cualquier lugar, por 250 dólares. "
                                      "Encuéntrala en Contoso Outdoors."),
    "fr": ("fr-FR", "fr-FR-Soleil", "De la pluie annoncée ? Qu'elle vienne. La TrailMaster X4 d'OutdoorLiving accueille quatre personnes, "
                                    "tient debout toute seule et inclut son double toit. Votre maison, partout, pour 250 dollars. "
                                    "En vente chez Contoso Outdoors."),
}
spots = {"en": radio}
for code_, (lang, voice, text) in RADIO.items():
    spots[code_] = speak(f"07-radio-{code_}", text, voice=voice, model=VOICE_HERO, style="excited", lang=lang)

play({f"Radio spot · {code_}": path for code_, path in spots.items()})
for code_, name in (("en", "05-radio-en"), ("es", "07-radio-es"), ("fr", "07-radio-fr")):
    if name in run_log:
        length = run_log[name]["audio_seconds"]
        print(f"   {code_}: {length} s {'✅' if length <= 15 else '❌ over 15 s'}")

🎙️ 07-radio-es · MAI-Voice-2.1 · first sound after 1.83 s · 20.7 s of audio
🎙️ 07-radio-fr · MAI-Voice-2.1 · first sound after 1.42 s · 15.8 s of audio
▶️ Radio spot · en


▶️ Radio spot · es


▶️ Radio spot · fr


   en: 16.2 s ❌ over 15 s
   es: 20.7 s ❌ over 15 s
   fr: 15.8 s ❌ over 15 s


👀 **What to look for:**

- [ ] Do the Spanish and French spots sound natural, not like an English speaker reading?
- [ ] Is *TrailMaster X4*, a brand name, said the same way in all three?
- [ ] Did a translation run longer? The same message often needs more words in another language.

⚠️ **The honest limit:** our Japanese poster from 01 can't get a matching MAI voice yet: the [MAI voices docs](https://learn.microsoft.com/azure/ai-services/speech-service/mai-voices) list no Japanese voice as of October 2026. Check the list for each market before you promise one.

🧪 **Try it yourself:** if a spot ran over 15 seconds, shorten that language's script and run the cell again.

<img src="./assets/divider.svg" alt="" width="100%" height="6" />

## 2.8 Does the hotline feel live?

<img src="./assets/asks/02-08.svg" alt="Your customer asks: Our setup hotline has to feel live. How fast does it start talking?" width="900" />
<br/>

On a phone line, people don't time the whole answer. They notice the **silence before it starts**. So let's measure *time to first sound* for both models, on the same short answer.

We send one warm-up request to each model first, and don't count it: the first call also sets up the connection. Then three timed rounds each.

> 🔮 **Guess first:** how much sooner will Flash start talking? Half the time? A tenth?

In [7]:
ANSWER = "Yes. The TrailMaster X4 comes with a rainfly, stakes, guy lines and a carry bag."

for model, tag in ((VOICE_HERO, "hero"), (VOICE_FAST, "flash")):
    speak(f"08-warmup-{tag}", ANSWER, model=model, style="customer_call_center")  # Not counted: the first call also connects.
    for round_ in (1, 2, 3):
        speak(f"08-hotline-{tag}-{round_}", ANSWER, model=model, style="customer_call_center")

print()
for model, tag in ((VOICE_HERO, "hero"), (VOICE_FAST, "flash")):
    rounds = [run_log[f"08-hotline-{tag}-{n}"] for n in (1, 2, 3) if f"08-hotline-{tag}-{n}" in run_log]
    if rounds:
        first = statistics.median(r["first_audio"] for r in rounds)
        total = statistics.median(r["total"] for r in rounds)
        print(f"{model:22} first sound: {first:.2f} s · whole answer ready: {total:.2f} s  (median of {len(rounds)})")
play({"Hotline · MAI-Voice-2.1": saved("08-hotline-hero-1"), "Hotline · MAI-Voice-2.1-Flash": saved("08-hotline-flash-1")})

🎙️ 08-warmup-hero · MAI-Voice-2.1 · first sound after 1.53 s · 5.8 s of audio
🎙️ 08-hotline-hero-1 · MAI-Voice-2.1 · first sound after 1.67 s · 5.7 s of audio
🎙️ 08-hotline-hero-2 · MAI-Voice-2.1 · first sound after 2.37 s · 6.3 s of audio
🎙️ 08-hotline-hero-3 · MAI-Voice-2.1 · first sound after 1.35 s · 6.4 s of audio
🎙️ 08-warmup-flash · MAI-Voice-2.1-Flash · first sound after 0.97 s · 6.6 s of audio
🎙️ 08-hotline-flash-1 · MAI-Voice-2.1-Flash · first sound after 1.31 s · 5.2 s of audio
🎙️ 08-hotline-flash-2 · MAI-Voice-2.1-Flash · first sound after 0.94 s · 6.6 s of audio
🎙️ 08-hotline-flash-3 · MAI-Voice-2.1-Flash · first sound after 1.11 s · 6.4 s of audio

MAI-Voice-2.1          first sound: 1.67 s · whole answer ready: 1.87 s  (median of 3)
MAI-Voice-2.1-Flash    first sound: 1.11 s · whole answer ready: 1.31 s  (median of 3)
▶️ Hotline · MAI-Voice-2.1


▶️ Hotline · MAI-Voice-2.1-Flash


👀 **What to look for:**

- [ ] Which model starts talking sooner? By how much, **in your region**?
- [ ] Listen to both. On a phone call, would you notice a difference in quality?
- [ ] Are the three rounds close to each other, or all over the place?

⚠️ **The honest limit:** three rounds from one place is a quick check, not a benchmark. Network and time of day move these numbers. Measure where your customers are, and look at the spread, not just the middle.

🧪 **Try it yourself:** change `ANSWER` to a two-sentence answer. Does the gap between the models change?

<img src="./assets/divider.svg" alt="" width="100%" height="6" />

## 2.9 Read the setup guide aloud

<img src="./assets/asks/02-09.svg" alt="Your customer asks: Can it read our setup guide aloud, clearly enough to follow along?" width="900" />
<br/>

Long audio is a different job: people listen while their hands are busy with tent poles. So clarity and steady pacing matter more than energy.

We read the six setup steps straight from the product manual in the shared dataset, `data/contoso-retail/manuals/trailmaster-x4-tent.md`.

> 🔮 **Guess first:** how long will six setup steps take to read?

In [8]:
manual = (DATA / "manuals" / "trailmaster-x4-tent.md").read_text(encoding="utf-8")
setup = manual.split("### Tent Setup", 1)[1].split("### Tent Takedown", 1)[0]
steps = []
for block in re.split(r"#### ", setup)[1:]:
    title, *bullets = [line.strip("- ").strip() for line in block.strip().splitlines() if line.strip()]
    steps.append(f"{title}. " + " ".join(bullets))
GUIDE = " ".join(steps)
print(f"📖 {len(steps)} steps, {len(GUIDE.split())} words\n")

guide = speak("09-setup-guide", GUIDE, voice="en-US-Harper", model=VOICE_HERO, style="narrator")
play({"Setup guide · MAI-Voice-2.1 · narrator": guide})

📖 6 steps, 218 words

🎙️ 09-setup-guide · MAI-Voice-2.1 · first sound after 1.91 s · 82.1 s of audio
▶️ Setup guide · MAI-Voice-2.1 · narrator


👀 **What to look for:**

- [ ] Can you tell where one step ends and the next begins?
- [ ] Is the pace slow enough to follow while you work?
- [ ] Does it say *"guy lines"* and *"rainfly"* clearly?

⚠️ **The honest limit:** the voice reads what the manual says, including *"Rainfly (if applicable)"*, even though the X4 always includes one. A voice model doesn't fix the source text. That's still your job.

🧪 **Try it yourself:** add `"Step complete. "` between steps, or try the `educational` style. Which is easier to follow?

<img src="./assets/divider.svg" alt="" width="100%" height="6" />

## 2.10 Both sides of a support call

<img src="./assets/asks/02-10.svg" alt="Your customer asks: We train support staff on example calls. Can it play both sides?" width="900" />
<br/>

New support staff learn from example calls. One SSML document can hold **two voices**: here, a customer (Grant) and an agent (Harper). The agent's answers come word for word from the manual's FAQ and warranty.

We'll use this call again in notebook 03, to see if transcription can tell who said what.

> 🔮 **Guess first:** will it sound like a conversation, or like two people reading?

In [9]:
CALL = [
    ("customer", "en-US-Grant", None, "Hi. Can I use the TrailMaster X4 in winter?"),
    ("agent", "en-US-Harper", "customer_call_center",
     "Good question. The TrailMaster X4 is designed for 3-season use, and may not be suitable for extreme winter conditions with heavy snow and freezing temperatures."),
    ("customer", "en-US-Grant", None, "Okay. And is there a warranty?"),
    ("agent", "en-US-Harper", "customer_call_center",
     "Yes. The TrailMaster X4 comes with a 2-year limited warranty against manufacturing defects."),
]
call = speak("10-support-call", " ".join(line for *_, line in CALL), model=VOICE_HERO,
             parts=[ssml(line, voice, VOICE_HERO, style) for _, voice, style, line in CALL],
             turns=[(who, line) for who, _, _, line in CALL])
play({"Support call · customer: Grant · agent: Harper": call})

🎙️ 10-support-call · MAI-Voice-2.1 · first sound after 2.15 s · 22.1 s of audio
▶️ Support call · customer: Grant · agent: Harper


👀 **What to look for:**

- [ ] Are the two speakers easy to tell apart?
- [ ] Does the agent sound helpful, not robotic?
- [ ] Are the facts exactly as the manual says: *3-season*, *2-year limited warranty*?

⚠️ **The honest limit:** real calls have interruptions, pauses and background noise. Generated calls are clean, which makes them good for training scripts and too easy as a test set. Notebook 03 comes back to this.

🧪 **Try it yourself:** add a third turn where the customer asks about the weight. The manual says 12 lbs.

<img src="./assets/divider.svg" alt="" width="100%" height="6" />

## 2.11 Which voice do you need?

<img src="./assets/asks/02-11.svg" alt="Your customer asks: Do we pay for the best voice every time, or is there a faster option?" width="900" />
<br/>

Here's everything we made, side by side. Look at the job, then at the numbers.

> 🔮 **Guess first:** which clips would you move to Flash without anyone noticing?

In [10]:
print(f"{'clip':26} {'model':20} {'first sound':>12} {'audio':>8}")
for name, entry in run_log.items():
    if name.startswith("08-warmup"):
        continue
    print(f"{name:26} {entry['model']:20} {entry['first_audio']:>10.2f} s {entry['audio_seconds']:>6.1f} s")

clip                       model                 first sound    audio
05-radio-en                MAI-Voice-2.1              1.68 s   16.2 s
06-mood-narrator           MAI-Voice-2.1              2.51 s    7.2 s
06-mood-excited            MAI-Voice-2.1              2.15 s    7.5 s
06-mood-customer_call_center MAI-Voice-2.1              1.56 s    5.7 s
07-radio-es                MAI-Voice-2.1              1.83 s   20.7 s
07-radio-fr                MAI-Voice-2.1              1.42 s   15.8 s
08-hotline-hero-1          MAI-Voice-2.1              1.67 s    5.7 s
08-hotline-hero-2          MAI-Voice-2.1              2.37 s    6.3 s
08-hotline-hero-3          MAI-Voice-2.1              1.35 s    6.4 s
08-hotline-flash-1         MAI-Voice-2.1-Flash        1.31 s    5.2 s
08-hotline-flash-2         MAI-Voice-2.1-Flash        0.94 s    6.6 s
08-hotline-flash-3         MAI-Voice-2.1-Flash        1.11 s    6.4 s
09-setup-guide             MAI-Voice-2.1              1.91 s   82.1 s
10-support-call   

| Pick **MAI Voice 2.1 Flash** when… | Pick **MAI Voice 2.1** when… |
| --- | --- |
| Someone is waiting: a hotline, a kiosk, a voice agent | It will be heard again and again: an ad, a guide |
| It's short and changes every time | It's long-form, like a guide or an audiobook |
| You're drafting scripts to time them | It's the final take |

**Not sure? Use both.** Draft and time your scripts on Flash, then record the final version on MAI Voice 2.1.

🗣️ **Four questions to ask a customer:**

1. Is someone waiting for this audio, live?
2. Will people hear it once, or many times?
3. How long is it: a sentence, or minutes?
4. Which languages and markets?

<img src="./assets/divider.svg" alt="" width="100%" height="6" />

## 2.12 Teach it to a customer

### What it did, and where to show it

| Job | Why a customer cares | Show them |
| --- | --- | --- |
| A timed radio spot | Ads that fit the slot | 2.5 |
| One voice, many moods | A brand voice that suits each moment | 2.6 |
| Native voices in other markets | Local ads without new voice actors | 2.7 |
| Fast first sound | Phone lines and agents that feel live | 2.8 |
| Long-form reading | Guides and training content | 2.9 |
| Several voices in one file | Example calls and dialogues | 2.10 |

### Honest answers to common questions

| They ask… | You can say… |
| --- | --- |
| Is it ready for production? | MAI Voice 2.1 and Flash are in **preview**. Pilot first, and check the docs for the latest status. |
| Does it do our language? | 23 languages, with a list of voices for each. Check that your market has a voice, and have a native speaker review. |
| Do we have to tell people it's AI? | Yes, be transparent. See the [transparency note](https://learn.microsoft.com/azure/ai-foundry/responsible-ai/speech-service/text-to-speech/transparency-note). |
| How fast is it? | Measure it in your region, like section 2.8. Don't quote someone else's numbers. |

### When something goes wrong

| You see | It usually means | Try |
| --- | --- | --- |
| `didn't finish` with an auth message | No role, or not signed in | `az login`; check the Foundry User role |
| `didn't finish` mentioning the voice | That voice or style isn't available | Check the voice and style list in the docs |
| Silence or a very short file | Empty or malformed SSML | Print `ssml(...)` and read it |

<img src="./assets/divider.svg" alt="" width="100%" height="6" />

## 2.13 Check yourself

1. A customer is building a voice agent for their store. Which model would you start with, and why?
2. Your Spanish radio spot runs 17 seconds. What do you change: the voice, the speed, or the script?
3. Why did we throw away the first, warm-up request in 2.8?

<details>
<summary>Show answers</summary>

1. **MAI Voice 2.1 Flash.** Someone is waiting, so the time to first sound matters most.
2. **The script.** Fewer words fit the slot without sounding rushed.
3. The first call also sets up the connection, so it's slower for reasons that have nothing to do with the model.

</details>

### 🤖 Ask your agent

- *"Rewrite this radio script to fit 12 seconds without losing the price or the product name."*
- *"Turn the TrailMaster X4 takedown steps into a script for a calm, step-by-step voice guide."*

<img src="./assets/divider.svg" alt="" width="100%" height="6" />

## 2.14 Cost and cleanup

This notebook doesn't create any Azure resources. You pay per character spoken, so a full run costs very little. When you're done:

- **Keep `output/02/`**: notebook 03 transcribes these clips. Git ignores the folder.
- **Delete `output/02/`** to clear your audio.

➡️ **Next:** [03 · Hear your customers](./03-hear-your-customers.ipynb). We transcribe these clips, and check every word against the scripts.